# Finding the Ground State of H₂ with a Variational Quantum Eigensolver: Everything from Scratch
**Series F: Quantum Meets Machine Learning, Post 2** &nbsp;|&nbsp; companion *primer* notebook

This is the lightweight companion to the full tutorial notebook `H2_VQE.ipynb`, which uses Qiskit Nature, Qiskit Algorithms and PySCF.
**Part I of this primer needs none of those; Part II installs them and runs the same pipeline.** It uses only NumPy and SciPy and re-derives, from scratch, every number the Qiskit tutorial prints: the STO-3G integrals, the fermionic Hamiltonian, its Jordan-Wigner qubit form with 15 Pauli strings, the five-term two-qubit Hamiltonian, the UCCSD ansatz, VQE, the potential energy curve, and the measurement-noise experiment. Part I runs on a CPU in under a minute.

| Section | What you derive and check |
|---|---|
| 1 | STO-3G integrals of H₂ from Gaussian formulas; Hartree-Fock energy |
| 2 | the 16×16 Jordan-Wigner Hamiltonian and its 15 Pauli strings |
| 3 | the five-term two-qubit Hamiltonian, in closed form |
| 4 | the variational principle, a one-parameter circuit, the correlation energy |
| 5 | a real UCCSD ansatz (3 parameters) and VQE; Brillouin's theorem |
| 6 | the potential energy curve, equilibrium, bond energy, why Hartree-Fock fails |
| 7 | how an energy is *measured*: variance and shots |
| 8 | the noise experiment, including a puzzle in the saved output of the Qiskit tutorial |
| **Part II** | the same pipeline with Qiskit Nature and PySCF, compared with Part I; a test of the estimator behind the tutorial's noise experiment |

**How to use it:** run top to bottom (**Runtime → Run all**), try the **Stop and predict** cells, then do the exercises.
Values printed by the Qiskit tutorial are typed in where we compare against them.

## 0. Setup

In [ ]:
import time, itertools, math
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from math import erf, sqrt, pi, exp
from scipy.linalg import expm
from scipy.optimize import minimize, minimize_scalar

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")

BOHR_PER_ANGSTROM = 1 / 0.529177210903       # 1 Angstrom in bohr
HARTREE_TO_KCAL = 627.5095                   # 1 Hartree in kcal/mol
HARTREE_TO_EV = 27.211386
CHEMICAL_ACCURACY = 1.0 / HARTREE_TO_KCAL    # 1 kcal/mol, in Hartree (1.59 mHa)
print(f"chemical accuracy = {1e3 * CHEMICAL_ACCURACY:.3f} mHartree")

## 1. The integrals, from Gaussians
In the minimal basis STO-3G each hydrogen contributes one function: a fixed combination of three normalised Gaussians, $\phi(\mathbf r)=\sum_{i=1}^3 d_i\,(2\alpha_i/\pi)^{3/4}e^{-\alpha_i|\mathbf r-\mathbf A|^2}$. For *s*-type Gaussians every integral has a closed form (Szabo and Ostlund, Appendix A) that uses only the **Boys function** $F_0(x)=\tfrac12\sqrt{\pi/x}\,\mathrm{erf}(\sqrt x)$. We need four kinds: overlap $S$, kinetic $T$, nuclear attraction $V$ and the two-electron repulsion integrals $(\mu\nu|\lambda\sigma)$.

In [ ]:
ALPHA = np.array([3.42525091, 0.62391373, 0.16885540])      # STO-3G exponents of hydrogen 1s
COEF  = np.array([0.15432897, 0.53532814, 0.44463454])      # contraction coefficients

F0 = lambda x: 1.0 if x < 1e-12 else 0.5 * sqrt(pi / x) * erf(sqrt(x))
norm = lambda a: (2 * a / pi) ** 0.75                       # normalisation of a primitive s Gaussian
d2 = lambda A, B: float(np.sum((A - B) ** 2))

def p_overlap(a, A, b, B):
    p = a + b; return (pi / p) ** 1.5 * exp(-a * b / p * d2(A, B))
def p_kinetic(a, A, b, B):
    p = a + b; r = a * b / p; return r * (3 - 2 * r * d2(A, B)) * (pi / p) ** 1.5 * exp(-r * d2(A, B))
def p_nuclear(a, A, b, B, C, Z=1.0):
    p = a + b; P = (a * A + b * B) / p
    return -Z * 2 * pi / p * exp(-a * b / p * d2(A, B)) * F0(p * d2(P, C))
def p_eri(a, A, b, B, c, C, d, D):
    p, q = a + b, c + d; P, Q = (a * A + b * B) / p, (c * C + d * D) / q
    return (2 * pi ** 2.5 / (p * q * sqrt(p + q)) * exp(-a * b / p * d2(A, B)) * exp(-c * d / q * d2(C, D))
            * F0(p * q / (p + q) * d2(P, Q)))

def integrals(R_angstrom):
    """Overlap S, one-electron matrix h = T + V, two-electron tensor (mu nu|lambda sigma) and nuclear repulsion, for H2."""
    R = R_angstrom * BOHR_PER_ANGSTROM
    cen = [np.zeros(3), np.array([0.0, 0.0, R])]                     # the two nuclei (and basis function centres)
    S, T, V = (np.zeros((2, 2)) for _ in range(3)); G = np.zeros((2, 2, 2, 2))
    for m, k in itertools.product(range(2), repeat=2):
        for (i, a), (j, b) in itertools.product(enumerate(ALPHA), repeat=2):
            w = COEF[i] * COEF[j] * norm(a) * norm(b)
            S[m, k] += w * p_overlap(a, cen[m], b, cen[k]); T[m, k] += w * p_kinetic(a, cen[m], b, cen[k])
            V[m, k] += w * sum(p_nuclear(a, cen[m], b, cen[k], C) for C in cen)
    for m, k, l, s in itertools.product(range(2), repeat=4):
        G[m, k, l, s] = sum(COEF[i] * COEF[j] * COEF[u] * COEF[v] * norm(a) * norm(b) * norm(c) * norm(d)
                            * p_eri(a, cen[m], b, cen[k], c, cen[l], d, cen[s])
                            for (i, a), (j, b), (u, c), (v, d) in itertools.product(enumerate(ALPHA), repeat=4))
    return S, T + V, G, 1.0 / R

def mo_integrals(R_angstrom):
    """Molecular-orbital integrals. By symmetry the orbitals are fixed: sigma_g = (1+2)/norm, sigma_u = (1-2)/norm."""
    S, h, G, Enuc = integrals(R_angstrom)
    s = S[0, 1]
    C = np.array([np.array([1, 1]) / sqrt(2 * (1 + s)), np.array([1, -1]) / sqrt(2 * (1 - s))]).T      # columns: g, u
    hmo = C.T @ h @ C
    eri = np.einsum("pi,qj,rk,sl,pqrs->ijkl", C, C, C, C, G)                                           # chemists' notation (pq|rs)
    return hmo, eri, Enuc, s

hmo, eri, Enuc, s12 = mo_integrals(0.735)
print(f"overlap S12 = {s12:.4f} | nuclear repulsion = {Enuc:.9f} Hartree (1/R with R = {0.735 * BOHR_PER_ANGSTROM:.4f} bohr)")
print("h_gg, h_uu        :", hmo[0, 0].round(8), hmo[1, 1].round(8), "  (PySCF/Qiskit: -1.2563391, -0.4718960; off-diagonal h_gu =", f"{hmo[0, 1]:.1e})")
print("(gg|gg)/2, (gg|uu)/2:", (eri[0, 0, 0, 0] / 2).round(8), (eri[0, 0, 1, 1] / 2).round(8), "  (Qiskit's fermionic operator: 0.33785508, 0.33229087)")
print("(gu|gu) = K        :", eri[0, 1, 0, 1].round(8))
E_hf = 2 * hmo[0, 0] + eri[0, 0, 0, 0] + Enuc          # closed-shell Hartree-Fock: both electrons in sigma_g
print(f"Hartree-Fock total energy = {E_hf:.6f} Hartree   (PySCF: -1.116999)")

## 2. The fermionic Hamiltonian and its Jordan-Wigner image
In second quantization (spin orbitals $p=0,1$ are $\sigma_g\alpha,\sigma_u\alpha$ and $p=2,3$ are $\sigma_g\beta,\sigma_u\beta$, as in Qiskit Nature)
$$H=\sum_{pq}h_{pq}\,a_p^\dagger a_q+\tfrac12\sum_{pqrs}(pq|rs)\,a_p^\dagger a_r^\dagger a_s a_q,$$
with spin conservation built into $h$ and $(pq|rs)$. The Jordan-Wigner mapping represents $a_p=Z\otimes\cdots\otimes Z\otimes\sigma^-\otimes I\cdots$ on 4 qubits, so $H$ becomes a $16\times16$ matrix: the whole Fock space (0 to 4 electrons).
**Stop and predict:** the Qiskit tutorial's Table 1 says this operator has **15 Pauli strings**. Which group of four strings (the double-excitation terms) should have coefficients related to the exchange integral $K$?

In [ ]:
spat = lambda p: p % 2; spin = lambda p: p // 2
Zm, Im, lower = np.diag([1.0, -1.0]), np.eye(2), np.array([[0.0, 1.0], [0.0, 0.0]])      # sigma^- maps |1> to |0>
def kron_all(ms):
    out = np.array([[1.0]])
    for m in ms: out = np.kron(out, m)
    return out

NQ = 4
a = [kron_all([Zm] * p + [lower] + [Im] * (NQ - p - 1)) for p in range(NQ)]       # Jordan-Wigner annihilation operators
ad = [x.T for x in a]
for p, q in itertools.product(range(NQ), repeat=2):                                # they obey the fermionic anticommutation rules
    assert np.allclose(a[p] @ ad[q] + ad[q] @ a[p], (p == q) * np.eye(16))

def fermionic_hamiltonian(hmo, eri):
    H1 = sum(hmo[spat(p), spat(q)] * (spin(p) == spin(q)) * ad[p] @ a[q] for p in range(NQ) for q in range(NQ))
    H2 = sum(0.5 * eri[spat(p), spat(q), spat(r), spat(s)] * (spin(p) == spin(q)) * (spin(r) == spin(s)) * ad[p] @ ad[r] @ a[s] @ a[q]
             for p, q, r, s in itertools.product(range(NQ), repeat=4))
    return H1 + H2

Hf = fermionic_hamiltonian(hmo, eri)
ev16 = np.linalg.eigvalsh(Hf)
print("lowest 4 eigenvalues of the 16x16 operator:", ev16[:4].round(6), "  (Qiskit tutorial: -1.857275 -1.256339 -1.256339 -1.244585)")

P1 = {"I": np.eye(2), "X": np.array([[0, 1], [1, 0]], float), "Y": np.array([[0, -1j], [1j, 0]]), "Z": Zm}
pauli_terms = {}
for lab in itertools.product("IXYZ", repeat=NQ):
    c = np.trace(kron_all([P1[ch] for ch in lab]).conj().T @ Hf) / 16
    if abs(c) > 1e-10: pauli_terms[("".join(lab))] = float(c.real)
print(f"\nnumber of Pauli strings: {len(pauli_terms)}   (Qiskit tutorial, Table 1: 15)")
print(pd.Series(pauli_terms).round(6).sort_values(key=abs, ascending=False).to_string())
K = eri[0, 1, 0, 1]
print(f"\nK/4 = {K / 4:.6f}: the coefficient of XXXX, XXYY, YYXX, YYYY")

Because $H$ conserves the number of electrons and the spin projection, we may restrict to the sector with **two electrons, one $\alpha$ and one $\beta$**. That sector is 4-dimensional (this is what the parity mapping with the *two-qubit reduction* exploits).

In [ ]:
Nop = sum(ad[p] @ a[p] for p in range(NQ))
Sz = 0.5 * (ad[0] @ a[0] + ad[1] @ a[1] - ad[2] @ a[2] - ad[3] @ a[3])
sector_idx = [i for i in range(16) if abs(Nop[i, i] - 2) < 1e-9 and abs(Sz[i, i]) < 1e-9]
ev_sector = np.linalg.eigvalsh(Hf[np.ix_(sector_idx, sector_idx)])
print("sector dimension:", len(sector_idx), "| its eigenvalues:", ev_sector.round(6))
print("Qiskit tutorial, exact eigenvalues of the 2-qubit operator: -1.857275 -1.244585 -0.882722 -0.224911")
one_e = [i for i in range(16) if abs(Nop[i, i] - 1) < 1e-9]
print("one-electron sector (H2+):", np.linalg.eigvalsh(Hf[np.ix_(one_e, one_e)]).round(6), " -> the value -1.256339 that appears twice in the 4-qubit spectrum")

## 3. The five-term two-qubit Hamiltonian, in closed form
Write the four determinants of the sector as $|\sigma_g\bar\sigma_g\rangle$ (Hartree-Fock), $|\sigma_u\bar\sigma_u\rangle$ (double excitation) and the two mixed ones. By parity ($g$ versus $u$), $H$ splits into two $2\times2$ blocks:
$$\begin{pmatrix}E_1&K\\K&E_2\end{pmatrix}\ \text{(gerade)},\qquad\begin{pmatrix}D&K\\K&D\end{pmatrix}\ \text{(ungerade)},$$
with $E_1=2h_{gg}+(gg|gg)$, $E_2=2h_{uu}+(uu|uu)$, $D=h_{gg}+h_{uu}+(gg|uu)$ and $K=(gu|gu)$. In the qubit basis $|q_1q_0\rangle$ with Hartree-Fock $=|01\rangle$ and the double excitation $=|10\rangle$, the diagonal matrix elements are $E_1,E_2,D,D$ for $|01\rangle,|10\rangle,|00\rangle,|11\rangle$ and $K$ couples $|01\rangle\leftrightarrow|10\rangle$ and $|00\rangle\leftrightarrow|11\rangle$. Decomposing into Pauli strings gives **five closed-form coefficients**.
**Stop and predict:** which Pauli string carries $K$?

In [ ]:
E1 = 2 * hmo[0, 0] + eri[0, 0, 0, 0]; E2 = 2 * hmo[1, 1] + eri[1, 1, 1, 1]
D = hmo[0, 0] + hmo[1, 1] + eri[0, 0, 1, 1]; K = eri[0, 1, 0, 1]
coef = {"II": (E1 + E2 + 2 * D) / 4, "IZ": (E2 - E1) / 4, "ZI": (E1 - E2) / 4, "ZZ": (2 * D - E1 - E2) / 4, "XX": K}
qiskit = {"II": -1.052373245772859, "IZ": 0.39793742484318045, "ZI": -0.39793742484318045, "ZZ": -0.01128010425623538, "XX": 0.18093119978423156}
print(pd.DataFrame({"derived from the integrals": coef, "Qiskit tutorial (Section 4)": qiskit,
                    "difference": {k: coef[k] - qiskit[k] for k in coef}}).round(9).to_string())

I2, X2, Z2 = np.eye(2), np.array([[0.0, 1.0], [1.0, 0.0]]), np.diag([1.0, -1.0])
op2 = lambda lab: np.kron({"I": I2, "X": X2, "Z": Z2}[lab[0]], {"I": I2, "X": X2, "Z": Z2}[lab[1]])      # Qiskit label "ab": a on qubit 1, b on qubit 0
H2q = sum(c * op2(l) for l, c in coef.items())
ev2 = np.linalg.eigvalsh(H2q)
print("\neigenvalues of the 2-qubit operator:", ev2.round(6), "| total ground-state energy:", round(ev2[0] + Enuc, 6))
assert np.allclose(ev2, ev_sector)                       # same spectrum as the sector of the 16x16 operator

print("\nthe four levels: ground state  (E1+E2)/2 - sqrt(((E1-E2)/2)^2 + K^2) =", round((E1 + E2) / 2 - sqrt(((E1 - E2) / 2) ** 2 + K ** 2), 6))
print("                 triplet         D - K                                  =", round(D - K, 6))
print("                 excited singlet D + K                                  =", round(D + K, 6))
print("                 excited singlet (E1+E2)/2 + sqrt(...)                  =", round((E1 + E2) / 2 + sqrt(((E1 - E2) / 2) ** 2 + K ** 2), 6))

## 4. The variational principle and a one-parameter circuit
For any normalised state $E=\langle\psi|H|\psi\rangle\ge E_0$. The exact ground state is a superposition of $|01\rangle$ and $|10\rangle$, so $|\psi(t)\rangle=\cos t\,|01\rangle+\sin t\,|10\rangle$ suffices. The circuit is: $X$ on qubit 0, a controlled-$R_y(2t)$ (control 0, target 1), and a CNOT (control 1, target 0). We simulate it with $4\times4$ matrices (basis index $=2q_1+q_0$).

In [ ]:
def circuit_state(t):
    X0 = np.kron(I2, X2)                                              # X on qubit 0 (the right factor)
    CRY = np.eye(4); c, s_ = np.cos(t), np.sin(t)                     # Ry(2t) on qubit 1, controlled on qubit 0 being 1
    CRY[np.ix_([1, 3], [1, 3])] = [[c, -s_], [s_, c]]
    CX = np.eye(4); CX[np.ix_([2, 3], [2, 3])] = [[0, 1], [1, 0]]     # NOT on qubit 0, controlled on qubit 1 being 1
    v = np.zeros(4); v[0] = 1.0
    return CX @ CRY @ X0 @ v

tt = np.linspace(-np.pi / 2, np.pi, 361)
sweep = np.array([circuit_state(t) @ H2q @ circuit_state(t) for t in tt])
by_hand = np.array([E1 * np.cos(t) ** 2 + E2 * np.sin(t) ** 2 + 2 * K * np.sin(t) * np.cos(t) for t in tt])
psi_check = max(np.abs(circuit_state(t) - np.array([0, np.cos(t), np.sin(t), 0])).max() for t in tt)
print(f"circuit state = cos t|01> + sin t|10>: max deviation {psi_check:.1e}")
print(f"E(t) from the circuit versus the closed form E1 cos^2 t + E2 sin^2 t + 2K sin t cos t: max deviation {np.abs(sweep - by_hand).max():.1e}")
assert sweep.min() >= ev2[0] - 1e-12                                  # the variational principle

t_star = 0.5 * np.arctan(2 * K / (E1 - E2))                          # stationary point: tan 2t = 2K / (E1 - E2)
E_min = (E1 + E2) / 2 - sqrt(((E1 - E2) / 2) ** 2 + K ** 2)
corr = E_min - E1
print(f"\nHartree-Fock energy (t = 0): {E1 + Enuc:.6f} | optimal t = {t_star:.4f} rad (Qiskit tutorial: -0.1118) | minimum: {E_min + Enuc:.6f}")
print(f"correlation energy E_exact - E_HF = {corr:.6f} Hartree = {corr * HARTREE_TO_KCAL:.2f} kcal/mol   (tutorial: -0.020307, -12.7)")

plt.figure(figsize=(6.5, 3.8))
plt.plot(tt, sweep + Enuc, lw=2, color=COLORS["blue"], label="E(t)")
plt.axhline(ev2[0] + Enuc, color="k", ls="--", lw=1, label="exact ground state")
plt.plot([0], [E1 + Enuc], "o", color=COLORS["red"], ms=8, label="Hartree-Fock (t = 0)"); plt.plot([t_star], [E_min + Enuc], "o", color=COLORS["green"], ms=8, label=f"minimum (t = {t_star:.2f})")
plt.ylim(ev2[0] + Enuc - 0.03, ev2[0] + Enuc + 0.6); plt.xlabel("t (radians)"); plt.ylabel("total energy (Hartree)"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

### VQE in a few lines
The optimiser is classical; the energy function is what the quantum computer would provide. Start at $t=0$ (Hartree-Fock).

In [ ]:
trace = []
def objective(x):
    e = circuit_state(x[0]) @ H2q @ circuit_state(x[0]); trace.append(e); return e
res = minimize(objective, x0=[0.0], method="COBYLA", tol=1e-8)
print(f"VQE energy {res.fun:.8f} | exact {ev2[0]:.8f} | error {abs(res.fun - ev2[0]):.1e} Hartree | optimal t {res.x[0]:.4f} | evaluations {len(trace)}")
print("(the Qiskit tutorial reports 43 evaluations; the count depends on the SciPy version)")

## 5. A real UCCSD ansatz, and VQE
Qiskit's `UCCSD` for H₂ has **3 parameters**: two single excitations ($\sigma_g\to\sigma_u$ for the $\alpha$ and for the $\beta$ electron) and one double excitation (both electrons at once). Each is a unitary $e^{\theta(T-T^\dagger)}$ with $T$ the excitation operator, applied to the Hartree-Fock state. We build exactly these in the 4-qubit Fock space.

In [ ]:
vac = np.zeros(16); vac[0] = 1.0
HF_state = ad[0] @ ad[2] @ vac                                        # both electrons in sigma_g (alpha and beta)
T1a, T1b = ad[1] @ a[0], ad[3] @ a[2]                                 # single excitations sigma_g -> sigma_u
T2 = ad[1] @ ad[3] @ a[2] @ a[0]                                      # the double excitation
G = [T - T.T for T in (T1a, T1b, T2)]                                 # anti-Hermitian generators

def uccsd_state(theta):
    return expm(theta[2] * G[2]) @ expm(theta[1] * G[1]) @ expm(theta[0] * G[0]) @ HF_state
def uccsd_energy(theta):
    v = uccsd_state(theta); return float(v @ Hf @ v)

print("E(theta = 0) =", round(uccsd_energy(np.zeros(3)) + Enuc, 6), "= Hartree-Fock")
print("<HF|H|single_alpha>, <HF|H|single_beta> :", f"{HF_state @ Hf @ (T1a @ HF_state):.1e}, {HF_state @ Hf @ (T1b @ HF_state):.1e}   (Brillouin's theorem)")
print("<HF|H|double>                           :", round(HF_state @ Hf @ (T2 @ HF_state), 6), "= K")
eps = 1e-6
grad0 = [(uccsd_energy(eps * np.eye(3)[k]) - uccsd_energy(-eps * np.eye(3)[k])) / (2 * eps) for k in range(3)]
print("gradient at theta = 0:", np.round(grad0, 6), f"  (2K = {2 * K:.6f})")

In [ ]:
history = []
def callback_obj(theta):
    e = uccsd_energy(theta); history.append(e); return e
res3 = minimize(callback_obj, np.zeros(3), method="L-BFGS-B")
print(f"UCCSD + L-BFGS-B from zero: energy {res3.fun + Enuc:.8f} | exact {ev2[0] + Enuc:.8f} | error {abs(res3.fun - ev2[0]):.1e} Hartree")
print(f"optimal parameters {np.round(res3.x, 4)} | energy evaluations {len(history)}   (the Qiskit tutorial: 3 parameters, 20 evaluations)")

fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.5), sharey=True)
grid = np.linspace(-np.pi, np.pi, 181)
for k in range(3):
    vals = [uccsd_energy(grid[i] * np.eye(3)[k]) + Enuc for i in range(len(grid))]
    ax[k].plot(grid, vals, lw=2, color=COLORS["blue"]); ax[k].axhline(ev2[0] + Enuc, color="k", ls="--", lw=1); ax[k].axhline(E1 + Enuc, color=COLORS["red"], ls=":", lw=1.5)
    ax[k].set(xlabel=f"parameter {k} (radians)", title=["single (alpha)", "single (beta)", "double"][k])
ax[0].set_ylabel("total energy (Hartree)"); plt.tight_layout(); plt.show()

Each single-excitation curve starts at the Hartree-Fock energy with zero slope and only goes *up* (to $D$), while the double excitation has slope $2K$ at the origin and reaches the exact energy: by parity, singles cannot couple to the Hartree-Fock state in H₂ (Brillouin's theorem), so all of the correlation comes from the double.

## 6. The potential energy curve
For every bond length we recompute the integrals, the Hartree-Fock energy, and the exact energy in the two-electron sector (the lowest of the four eigenvalues). The Qiskit tutorial's table of 14 distances is typed in for comparison.

In [ ]:
dist = [0.3, 0.4, 0.5, 0.6, 0.7, 0.735, 0.8, 0.9, 1.0, 1.2, 1.5, 1.8, 2.1, 2.5]
nb_hf = [-0.593828, -0.904361, -1.042996, -1.101128, -1.117349, -1.116999, -1.110850, -1.091914, -1.066109, -1.005107, -0.910874, -0.828848, -0.764178, -0.702944]
nb_ex = [-0.601804, -0.914150, -1.055160, -1.116286, -1.136189, -1.137306, -1.134148, -1.120560, -1.101150, -1.056741, -0.998149, -0.961817, -0.944375, -0.936055]

def energies(R):
    hmo_, eri_, Enuc_, _ = mo_integrals(R)
    e1 = 2 * hmo_[0, 0] + eri_[0, 0, 0, 0]; e2 = 2 * hmo_[1, 1] + eri_[1, 1, 1, 1]
    d_ = hmo_[0, 0] + hmo_[1, 1] + eri_[0, 0, 1, 1]; k_ = eri_[0, 1, 0, 1]
    w, v = np.linalg.eigh(np.array([[e1, k_], [k_, e2]]))
    return e1 + Enuc_, min(w[0], d_ - k_) + Enuc_, v[1, 0] ** 2          # HF, exact (FCI), weight of the double excitation

pes = np.array([energies(R) for R in dist])
cmp_ = pd.DataFrame({"R (A)": dist, "HF": pes[:, 0], "HF (Qiskit)": nb_hf, "exact": pes[:, 1], "exact (Qiskit)": nb_ex, "weight of sigma_u^2": pes[:, 2]})
print(cmp_.round(6).to_string(index=False))
print(f"\nlargest difference to the Qiskit table: HF {np.abs(pes[:, 0] - nb_hf).max():.1e}, exact {np.abs(pes[:, 1] - nb_ex).max():.1e} Hartree (the table has 6 digits)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
Rf = np.linspace(0.3, 2.5, 80); pf = np.array([energies(R) for R in Rf])
ax[0].plot(Rf, pf[:, 0], "--", color=COLORS["red"], label="Hartree-Fock"); ax[0].plot(Rf, pf[:, 1], color="k", lw=2, label="exact (this basis)")
ax[0].plot(dist, nb_ex, "o", color=COLORS["blue"], ms=5, label="Qiskit tutorial (VQE = exact)")
ax[0].set(xlabel="H-H distance (Angstrom)", ylabel="total energy (Hartree)", title="(a) Potential energy curve of H2 (STO-3G)"); ax[0].legend(fontsize=8)
ax[1].semilogy(Rf, 1e3 * (pf[:, 0] - pf[:, 1]), color=COLORS["red"]); ax[1].axhline(1e3 * CHEMICAL_ACCURACY, color=COLORS["green"], ls="--", label="chemical accuracy")
ax[1].set(xlabel="H-H distance (Angstrom)", ylabel="Hartree-Fock error (mHartree)", title="(b) Hartree-Fock error"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

# equilibrium and bond energy
opt = minimize_scalar(lambda R: energies(R)[1], bounds=(0.6, 0.9), method="bounded", options=dict(xatol=1e-6))
EH = sum(COEF[i] * COEF[j] * norm(a_) * norm(b_) * (p_kinetic(a_, np.zeros(3), b_, np.zeros(3)) + p_nuclear(a_, np.zeros(3), b_, np.zeros(3), np.zeros(3)))
         for (i, a_), (j, b_) in itertools.product(enumerate(ALPHA), repeat=2))
De = 2 * EH - opt.fun
print(f"equilibrium bond length (exact, STO-3G): {opt.x:.4f} A | energy {opt.fun:.6f} Hartree")
print(f"hydrogen atom energy in STO-3G: {EH:.6f} Hartree | bond energy De = {De:.6f} Hartree = {De * HARTREE_TO_EV:.2f} eV = {De * HARTREE_TO_KCAL:.1f} kcal/mol")
far = energies(20.0)
print(f"\nat 20 A: Hartree-Fock {far[0]:.4f}, exact {far[1]:.4f}, two separate atoms {2 * EH:.4f} -> Hartree-Fock misses by {far[0] - 2 * EH:.3f} Hartree = {(far[0] - 2 * EH) * HARTREE_TO_KCAL:.0f} kcal/mol")
print("weight of the doubly excited configuration in the exact ground state:",
      {R: round(energies(R)[2], 4) for R in (0.5, 0.735, 1.0, 1.5, 2.5, 5.0)}, "-> 0.5 at large R: static correlation")

## 7. How is an energy measured?
On a quantum computer the energy $\langle H\rangle=\sum_kc_k\langle P_k\rangle$ is *estimated* from repeated measurements. Terms that commute can share a measurement setting: here $\{II,IZ,ZI,ZZ\}$ are all diagonal (one setting, the $Z$ basis) and $XX$ needs a second setting (the $X$ basis). For the state $\cos t|01\rangle+\sin t|10\rangle$ (derivations in the post's supplement):
$$\mathrm{Var}_Z=\tfrac14\sin^22t\,(E_1-E_2)^2,\qquad \mathrm{Var}_X=K^2\cos^22t,$$
and with $S$ shots per setting the standard error of the estimated energy is $\sqrt{(\mathrm{Var}_Z+\mathrm{Var}_X)/S}$.
**Stop and predict:** at the Hartree-Fock point $t=0$, which setting is deterministic, and how many shots does the other one need for chemical accuracy?

In [ ]:
var_Z = lambda t: 0.25 * np.sin(2 * t) ** 2 * (E1 - E2) ** 2
var_X = lambda t: K ** 2 * np.cos(2 * t) ** 2
print(f"at Hartree-Fock (t=0):  Var_Z = {var_Z(0.0):.4f}, Var_X = {var_X(0.0):.4f}  -> the Z setting is exact; XX alone needs {var_X(0.0) / CHEMICAL_ACCURACY ** 2:,.0f} shots")
print(f"at the optimum t*:      Var_Z = {var_Z(t_star):.5f}, Var_X = {var_X(t_star):.5f}  (equal at a stationary point) -> {(var_Z(t_star) + var_X(t_star)) / CHEMICAL_ACCURACY ** 2:,.0f} shots per setting")

print("\nshots per setting needed at t* for a given precision p of the energy:")
precisions = [0.1, 0.03, 0.01, 0.003, 0.001, 0.0003]
for p in precisions:
    print(f"   p = {p:7.4f} Hartree -> {(var_Z(t_star) + var_X(t_star)) / p ** 2:>10,.0f} shots per setting")

# Monte-Carlo check of the formulas: sample the two settings and compare with theory
rng = np.random.default_rng(1)
def measured_energy(t, shots, rng):
    n01 = rng.binomial(shots, np.cos(t) ** 2)                                  # Z setting: outcome |01> (energy E1) or |10> (energy E2)
    z_part = (n01 * E1 + (shots - n01) * E2) / shots
    x_part = K * (2 * rng.binomial(shots, (1 + np.sin(2 * t)) / 2) - shots) / shots     # X setting: XX = +1 with probability (1 + sin 2t)/2
    return z_part + x_part
draws = np.array([measured_energy(t_star, 1000, rng) for _ in range(20000)])
print(f"\n1000 shots per setting at t*: mean of the estimates {draws.mean():.6f} (exact {E_min:.6f}), std {draws.std():.5f} (theory {np.sqrt((var_Z(t_star) + var_X(t_star)) / 1000):.5f})")

## 8. How precise must the energies be? The noise experiment
We minimise $E(t)$ with COBYLA (the settings of the Qiskit tutorial: start at $t=0$, `maxiter=60`, `rhobeg=0.3`), where every energy evaluation carries a random error of size $p$. At the end we evaluate the **exact** energy of the optimised $t$. Three ways of adding the noise:

* **independent**: a fresh random error at *every* evaluation (this is what finite sampling does);
* **same draw each call**: the same error is added at every evaluation (a model of an estimator that re-creates its random generator from the same seed on every call);
* **none**.

The saved output of the Qiskit tutorial's Section 7 reads *"mean error 0.000 mHa, 100 % of runs within chemical accuracy"* for **all** precisions, from 0.1 down to 0.0003 Hartree. **Stop and predict:** which of the three models can produce that?

In [ ]:
N_REPEATS = 200
def noisy_run(p, rep, model):
    exact = lambda x: circuit_state(x[0]) @ H2q @ circuit_state(x[0])
    if model == "independent":
        rng = np.random.default_rng(10_000 + rep); f = lambda x: exact(x) + rng.normal(0.0, p)
    elif model == "same draw each call":
        f = lambda x: exact(x) + np.random.default_rng(10_000 + rep).normal(0.0, p)     # a new generator with the SAME seed on every call
    else:
        f = exact
    r = minimize(f, [0.0], method="COBYLA", options=dict(maxiter=60, rhobeg=0.3))
    return exact(r.x) - ev2[0]

t0 = time.perf_counter(); table_noise = {}
for model in ("independent", "same draw each call", "none"):
    rows = []
    n_rep = N_REPEATS if model == "independent" else 20            # the other two models are (nearly) deterministic
    for p in precisions:
        e = np.array([noisy_run(p, rep, model) for rep in range(n_rep)])
        rows.append((p, 1e3 * e.mean(), 1e3 * np.median(e), 100 * np.mean(e < CHEMICAL_ACCURACY), e.max()))
    table_noise[model] = pd.DataFrame(rows, columns=["precision p", "mean error (mHa)", "median error (mHa)", "within chem. accuracy (%)", "max error (Ha)"])
    print(f"--- noise model: {model} ({n_rep} repeats per precision)"); print(table_noise[model].round(4).to_string(index=False)); print()
print(f"done in {time.perf_counter() - t0:.0f} s")

In [ ]:
ind = table_noise["independent"]
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4))
ax[0].loglog(ind["precision p"], ind["mean error (mHa)"], "o-", color=COLORS["blue"], label="independent noise (mean of 200 runs)")
ax[0].loglog(ind["precision p"], 0.35 * 1e3 * ind["precision p"], ":", color=COLORS["gray"], label="0.35 x p")
ax[0].axhline(1e3 * CHEMICAL_ACCURACY, color=COLORS["green"], ls="--", label="chemical accuracy")
ax[0].axhline(4.4e-7, color=COLORS["red"], ls="-.", label="Qiskit tutorial's saved output (4e-7)")
ax[0].set(xlabel="precision p of each energy (Hartree)", ylabel="error of the final energy (mHartree)", title="(a) Error after optimisation"); ax[0].legend(fontsize=7)
ax[1].semilogx(ind["precision p"], ind["within chem. accuracy (%)"], "o-", color=COLORS["orange"], label="independent noise")
ax[1].axhline(100, color=COLORS["red"], ls="-.", lw=1, label="Qiskit tutorial's saved output")
ax[1].set(xlabel="precision p of each energy (Hartree)", ylabel="runs within chemical accuracy (%)", title="(b) Success rate", ylim=(-2, 105)); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

ratio = (ind["mean error (mHa)"] / (1e3 * ind["precision p"])).round(2).tolist()
print("mean error divided by p for the six precisions:", ratio, "-> the final error is roughly 0.3 to 0.4 times the noise level")

### The same experiment with real shot sampling
Instead of adding Gaussian noise we can *sample* the two measurement settings of Section 7 with $S$ shots each. The noise is then the true statistical noise of the estimator.

In [ ]:
def shot_run(S, rep):
    rng = np.random.default_rng(10_000 + rep)
    f = lambda x: measured_energy(x[0], S, rng)                       # a fresh sample at every evaluation
    r = minimize(f, [0.0], method="COBYLA", options=dict(maxiter=60, rhobeg=0.3))
    return circuit_state(r.x[0]) @ H2q @ circuit_state(r.x[0]) - ev2[0]

rows = []
for S in (10, 100, 1000, 10_000, 100_000, 1_000_000):
    p_eff = np.sqrt((var_Z(t_star) + var_X(t_star)) / S)
    e = np.array([shot_run(S, rep) for rep in range(N_REPEATS)])
    rows.append((S, p_eff, 1e3 * e.mean(), 1e3 * np.median(e), 100 * np.mean(e < CHEMICAL_ACCURACY)))
shots_table = pd.DataFrame(rows, columns=["shots per setting", "equivalent p (Hartree)", "mean error (mHa)", "median error (mHa)", "within chem. accuracy (%)"])
print(shots_table.round(4).to_string(index=False))

### A diagnostic for your own estimator
If the noise in an estimator were truly independent from call to call, evaluating it twice at the same parameter would give two *different* numbers. Here are the three models:

Part II runs this diagnostic on Qiskit's own `StatevectorEstimator`.

In [ ]:
def make_objective(p, model, seed=17):
    exact = lambda x: circuit_state(x[0]) @ H2q @ circuit_state(x[0])
    if model == "independent":
        rng = np.random.default_rng(seed); return lambda x: exact(x) + rng.normal(0.0, p)
    if model == "same draw each call":
        return lambda x: exact(x) + np.random.default_rng(seed).normal(0.0, p)
    return exact

for model in ("independent", "same draw each call", "none"):
    f = make_objective(0.01, model)
    v = [f([0.3]) for _ in range(3)]
    print(f"{model:22s} three evaluations at t = 0.3: {np.round(v, 5).tolist()}   all different: {len(set(np.round(v, 12))) == 3}")
print("\nA constant offset does not move the minimiser: with the 'same draw each call' model the optimiser sees E(t) + const and finds the exact minimum.")
print("If your noisy energies are identical at identical parameters, your noise is not noise: give each evaluation its own random draw.")

## 9. Your turn
**Exercise 1 (another geometry).** Recompute the five Pauli coefficients at $R=1.5$ Å (Section 3). Which coefficient changes most? How does the weight of the double excitation compare with 0.735 Å?

**Exercise 2 (qubit counting).** In the 6-31G basis each hydrogen has two $s$ functions. How many spatial orbitals, spin orbitals, Jordan-Wigner qubits and qubits after the two-qubit reduction does H₂ need?

**Exercise 3 (shots).** The Qiskit tutorial's Exercise 6 asks for the number of shots per energy evaluation needed for chemical accuracy at the Hartree-Fock point. Answer it with Section 7, and then for the optimum.

**Exercise 4 (the noise rule of thumb).** Section 8 suggests that the mean final error is roughly $0.3$ to $0.4$ times $p$. Use it to predict the mean error at $p=2$ mHa (as a range), then simulate it. How good is the rule?

**Exercise 5 (diagnosing an estimator).** You are given a noisy estimator as a black box. Design a test that tells you whether its noise is independent from call to call, and say what you would change if it is not.

**Exercise 6 (bond energy).** Use `energies` and the hydrogen-atom energy to compute the bond energy in the STO-3G basis. The experimental value is about 4.75 eV: is the minimal basis too generous or too stingy?

In [ ]:
# Starters
# Exercise 1
hm, er, En, _ = mo_integrals(1.5)
e1, e2 = 2 * hm[0, 0] + er[0, 0, 0, 0], 2 * hm[1, 1] + er[1, 1, 1, 1]; d_, k_ = hm[0, 0] + hm[1, 1] + er[0, 0, 1, 1], er[0, 1, 0, 1]
print("R = 1.5 A:", {"II": (e1 + e2 + 2 * d_) / 4, "IZ": (e2 - e1) / 4, "ZI": (e1 - e2) / 4, "ZZ": (2 * d_ - e1 - e2) / 4, "XX": k_})

# Exercise 2: counting
n_spatial = 2 * 2; print("6-31G: spatial orbitals", n_spatial, "| spin orbitals / JW qubits", 2 * n_spatial, "| after the two-qubit reduction", 2 * n_spatial - 2)

# Exercise 3
print("shots for chemical accuracy: Hartree-Fock point", round(var_X(0.0) / CHEMICAL_ACCURACY ** 2), "| optimum", round((var_Z(t_star) + var_X(t_star)) / CHEMICAL_ACCURACY ** 2))

# Exercise 4: predicted mean error at p = 2 mHa
print("predicted mean error at p = 2 mHa:", round(0.3 * 2, 2), "to", round(0.4 * 2, 2), "mHa")
sim = np.array([noisy_run(0.002, rep, "independent") for rep in range(N_REPEATS)])
print("simulated mean error at p = 2 mHa :", round(1e3 * sim.mean(), 2), "mHa")

## 10. Self-check

In [ ]:
assert abs(E_hf - (-1.116999)) < 2e-6 and abs(hmo[0, 0] + 1.2563391) < 1e-6 and abs(hmo[1, 1] + 0.4718960) < 1e-6
assert abs(eri[0, 0, 0, 0] / 2 - 0.33785508) < 1e-6 and abs(eri[0, 0, 1, 1] / 2 - 0.33229087) < 1e-6
assert len(pauli_terms) == 15 and np.allclose(ev16[:4], [-1.857275, -1.256339, -1.256339, -1.244585], atol=2e-6)
assert max(abs(coef[k] - qiskit[k]) for k in coef) < 2e-8 and np.allclose(ev2, [-1.857275, -1.244585, -0.882722, -0.224911], atol=2e-6)
assert abs(t_star + 0.1118) < 1e-4 and abs(corr + 0.020307) < 2e-6 and abs(res.fun - ev2[0]) < 1e-8
assert abs(HF_state @ Hf @ (T1a @ HF_state)) < 1e-12 and abs(grad0[2] - 2 * K) < 1e-6 and max(abs(grad0[0]), abs(grad0[1])) < 1e-6 and abs(res3.fun - ev2[0]) < 1e-7
assert np.abs(pes[:, 0] - nb_hf).max() < 1e-6 and np.abs(pes[:, 1] - nb_ex).max() < 1e-6 and abs(opt.x - 0.735) < 2e-3
assert abs(draws.std() - np.sqrt((var_Z(t_star) + var_X(t_star)) / 1000)) < 2e-4 and abs(var_Z(t_star) - var_X(t_star)) < 1e-12
assert ind["mean error (mHa)"].iloc[0] > 10 and ind["mean error (mHa)"].iloc[-1] < 0.5 and ind["within chem. accuracy (%)"].iloc[-1] > 95 and ind["within chem. accuracy (%)"].iloc[0] < 25
assert shots_table["mean error (mHa)"].iloc[0] > 10 and shots_table["within chem. accuracy (%)"].iloc[-1] > 95 and shots_table["mean error (mHa)"].is_monotonic_decreasing
assert table_noise["same draw each call"]["max error (Ha)"].max() < 1e-6 and table_noise["none"]["max error (Ha)"].max() < 1e-6
print("All checks passed: you reproduced the numbers in the post.")

# Part II. The same pipeline in Qiskit Nature
> **Read this first.** Part II installs **Qiskit**, **Qiskit Nature**, **Qiskit Algorithms** and **PySCF** (a few minutes the first time; restart the session if Colab asks) and runs the pipeline of the tutorial: PySCF computes the integrals, Qiskit Nature builds the fermionic operator and maps it to qubits, and VQE with the UCCSD ansatz finds the ground state. Every stage is compared with the number that Part I derived from scratch, so a disagreement points at one of the two implementations. The last two cells test the estimator behind the tutorial's noise experiment (Section 8 of Part I): **does a seeded `StatevectorEstimator` add the same noise at every call?**
>
> Each stage is guarded: if an interface of an installed package differs from the one used here, the cell prints what failed and the other stages still run. The last cell lists the checks that passed.

In [ ]:
!pip install -q qiskit qiskit-nature qiskit-algorithms pyscf pylatexenc

In [ ]:
import importlib.util
HAVE_QISKIT = all(importlib.util.find_spec(m) is not None for m in ("qiskit", "qiskit_nature", "qiskit_algorithms", "pyscf"))
checks = []                                     # (description, passed); summarized in the last cell of Part II
def check(label, ok):
    checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
if HAVE_QISKIT:
    import qiskit, qiskit_nature, qiskit_algorithms, pyscf
    from pyscf import gto, scf
    from qiskit import QuantumCircuit
    from qiskit.circuit import Parameter
    from qiskit.primitives import StatevectorEstimator
    from qiskit.quantum_info import SparsePauliOp
    from qiskit_nature.units import DistanceUnit
    from qiskit_nature.second_q.drivers import PySCFDriver
    from qiskit_nature.second_q.mappers import JordanWignerMapper, ParityMapper, BravyiKitaevMapper
    from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
    from qiskit_nature.second_q.algorithms import GroundStateEigensolver
    from qiskit_algorithms import VQE, NumPyMinimumEigensolver
    from qiskit_algorithms.optimizers import L_BFGS_B
    print(f"qiskit {qiskit.__version__} | qiskit-nature {qiskit_nature.__version__} | qiskit-algorithms {getattr(qiskit_algorithms, '__version__', '?')} | pyscf {pyscf.__version__}")
else:
    print("Qiskit Nature, Qiskit Algorithms or PySCF is not available: run the install cell above (restart the session if Colab asks), then re-run this cell.")

In [ ]:
# 1) PySCF and the problem, against the integrals of Part I
if HAVE_QISKIT:
    R = 0.735
    try:
        problem = PySCFDriver(atom=f"H 0.0 0.0 0.0; H 0.0 0.0 {R}", basis="sto3g", unit=DistanceUnit.ANGSTROM).run()
        check("1 alpha and 1 beta electron in 2 spatial orbitals (4 spin orbitals)", tuple(problem.num_particles) == (1, 1) and problem.num_spatial_orbitals == 2)
        E_nuc_q = problem.hamiltonian.nuclear_repulsion_energy
        check(f"nuclear repulsion {E_nuc_q:.6f} Hartree equals Part I ({Enuc:.6f})", abs(E_nuc_q - Enuc) < 1e-6)
        h_q = np.asarray(problem.hamiltonian.electronic_integrals.alpha["+-"])
        check("one-electron integrals h_gg and h_uu equal Part I (-1.25634, -0.47190); the off-diagonal element vanishes", abs(h_q[0, 0] - hmo[0, 0]) < 1e-6 and abs(h_q[1, 1] - hmo[1, 1]) < 1e-6 and abs(h_q[0, 1]) < 1e-8)
        E_hf_pyscf = float(scf.RHF(gto.M(atom=f"H 0 0 0; H 0 0 {R}", basis="sto3g", unit="Angstrom", verbose=0)).run().e_tot)
        check(f"PySCF Hartree-Fock total energy {E_hf_pyscf:.6f} equals Part I ({E_hf:.6f})", abs(E_hf_pyscf - E_hf) < 1e-6)
        f_op = problem.hamiltonian.second_q_op()
        check(f"fermionic operator with {len(f_op)} terms (Part I: 36)", len(f_op) == 36)
    except Exception as e:
        print("stage 1 could not be completed:", type(e).__name__, e)

In [ ]:
# 2) mapping to qubits: the four mappers, and the reduced operator against the five coefficients derived in Part I
if HAVE_QISKIT:
    try:
        mappers = {"Jordan-Wigner": JordanWignerMapper(), "Bravyi-Kitaev": BravyiKitaevMapper(), "Parity": ParityMapper(),
                   "Parity + 2-qubit reduction": ParityMapper(num_particles=problem.num_particles)}
        qubit_ops = {k: m.map(f_op).simplify() for k, m in mappers.items()}
        print(pd.DataFrame([(k, o.num_qubits, len(o), 2 ** o.num_qubits) for k, o in qubit_ops.items()], columns=["mapper", "qubits", "Pauli strings", "state-space dimension"]).to_string(index=False))
        jw, red = qubit_ops["Jordan-Wigner"], qubit_ops["Parity + 2-qubit reduction"]
        check("Jordan-Wigner: 4 qubits and 15 Pauli strings (Part I)", jw.num_qubits == 4 and len(jw) == 15)
        check("reduced operator: 2 qubits and 5 Pauli strings (Part I)", red.num_qubits == 2 and len(red) == 5)
        q_coef = {lab: float(np.real(c)) for lab, c in red.to_list()}
        print("\ncoefficients from Qiskit Nature:", {k: round(v, 6) for k, v in q_coef.items()}); print("coefficients derived in Part I :", {k: round(float(v), 6) for k, v in coef.items()})
        diff = max(abs(q_coef.get(k, 0.0) - coef[k]) for k in coef)
        check(f"the five coefficients equal those derived from the integrals in Part I (largest difference {diff:.1e})", diff < 1e-7)
        check("the four eigenvalues of the reduced operator equal Part I's", np.abs(np.linalg.eigvalsh(red.to_matrix()) - ev2).max() < 1e-6)
    except Exception as e:
        print("stage 2 could not be completed:", type(e).__name__, e)

In [ ]:
# 3) VQE with the UCCSD ansatz, against the exact energy of Part I
if HAVE_QISKIT:
    try:
        mapper = mappers["Parity + 2-qubit reduction"]
        init_state = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
        ansatz = UCCSD(problem.num_spatial_orbitals, problem.num_particles, mapper, initial_state=init_state)
        check(f"UCCSD ansatz: {ansatz.num_qubits} qubits, {ansatz.num_parameters} parameters (Part I: 2 and 3)", ansatz.num_qubits == 2 and ansatz.num_parameters == 3)
        evals = []
        vqe = VQE(StatevectorEstimator(), ansatz, L_BFGS_B(), callback=lambda count, params, mean, meta: evals.append(count)); vqe.initial_point = np.zeros(ansatz.num_parameters)
        res = GroundStateEigensolver(mapper, vqe).solve(problem)
        try: E_vqe = float(np.real(res.total_energies[0]))
        except Exception: E_vqe = float(np.real(res.eigenvalues[0])) + E_nuc_q
        E_ex = float(ev2[0] + Enuc)
        print(f"VQE total energy {E_vqe:.6f} | exact energy of Part I {E_ex:.6f} | difference {1e3 * (E_vqe - E_ex):+.5f} mHa | {len(evals)} energy evaluations")
        check("VQE equals the exact energy of Part I to 1e-6 Hartree", abs(E_vqe - E_ex) < 1e-6)
        check(f"... and the error is below chemical accuracy ({1e3 * CHEMICAL_ACCURACY:.2f} mHa)", abs(E_vqe - E_ex) < CHEMICAL_ACCURACY)
        check(f"fraction of the correlation energy recovered: {100 * (E_vqe - E_hf) / (E_ex - E_hf):.4f} %", abs((E_vqe - E_hf) / (E_ex - E_hf) - 1) < 1e-4)
    except Exception as e:
        print("stage 3 could not be completed:", type(e).__name__, e)

In [ ]:
# 4) the one-parameter circuit of Part I in Qiskit, with the Hamiltonian that Part I derived
if HAVE_QISKIT:
    try:
        t_par = Parameter("t"); toy = QuantumCircuit(2); toy.x(0); toy.cry(2 * t_par, 0, 1); toy.cx(1, 0)
        H_toy = SparsePauliOp.from_list(list(coef.items()))                         # the five coefficients derived in Part I
        energy = lambda x, est=StatevectorEstimator(): float(np.real(est.run([(toy, H_toy, np.atleast_1d(np.asarray(x, dtype=float)))]).result()[0].data.evs))
        grid = tt[::12]; q_energies = np.array([energy([x]) for x in grid]); my_energies = np.array([circuit_state(x) @ H2q @ circuit_state(x) for x in grid])
        check(f"Qiskit's circuit energies equal Part I's NumPy energies on {len(grid)} angles (largest difference {np.abs(q_energies - my_energies).max():.1e})", np.abs(q_energies - my_energies).max() < 1e-10)
        check("E(t = 0) is the Hartree-Fock energy of Part I", abs(energy([0.0]) + Enuc - E_hf) < 1e-6)
        check("E(t*) is the exact ground-state energy of Part I", abs(energy([t_star]) - ev2[0]) < 1e-8)
    except Exception as e:
        print("stage 4 could not be completed:", type(e).__name__, e)

### The diagnostic, on Qiskit's own estimator
The noise experiment of the tutorial creates every estimator as `StatevectorEstimator(default_precision=p, seed=...)`. Part I showed that if such an estimator adds **the same** noise value at every call, the optimizer sees $E(t)+\text{constant}$ and finds the exact minimum, which is what the tutorial's saved output shows. We now ask the estimator itself: five evaluations of the energy at the same parameter, in five ways.

In [ ]:
if HAVE_QISKIT:
    try:
        x0, p = [0.3], 0.01
        def five(kind):
            shared = StatevectorEstimator(default_precision=p, seed=1) if kind == "seeded, one estimator" else StatevectorEstimator(default_precision=p)
            out = []
            for i in range(5):
                if kind == "exact": e = StatevectorEstimator()
                elif kind == "seeded, new estimator per call": e = StatevectorEstimator(default_precision=p, seed=1)
                elif kind == "new seed for every call": e = StatevectorEstimator(default_precision=p, seed=100 + i)
                else: e = shared
                out.append(energy(x0, e))
            return np.array(out)
        kinds = ["exact", "seeded, one estimator", "seeded, new estimator per call", "unseeded, one estimator", "new seed for every call"]
        vals = {k: five(k) for k in kinds}; distinct = {k: len(set(np.round(v, 12))) for k, v in vals.items()}
        print(f"five evaluations of the energy at t = 0.3 (exact {energy(x0):.5f}); precision {p} Hartree where noise is requested\n")
        for k in kinds: print(f"  {k:32s} {np.round(vals[k], 5).tolist()}   distinct values: {distinct[k]}")
        sd = np.std([energy(x0, StatevectorEstimator(default_precision=p, seed=1000 + i)) for i in range(200)])
        print(f"\nspread of 200 independently seeded noisy energies: {sd:.4f} Hartree (requested precision {p})")
        check("a new seed for every call gives five different energies", distinct["new seed for every call"] == 5)
        check("the exact estimator gives one value", distinct["exact"] == 1)
        print("\nfinding for a SEEDED estimator used for several calls:")
        if distinct["seeded, one estimator"] == 1:
            print("   the same value at every call: its noise is a constant offset, the behaviour that explains the flat saved output of the tutorial's Figure 6.")
            print("   Fix: do not seed the estimator, or give every evaluation its own seed (the last row).")
        elif distinct["seeded, one estimator"] == 5:
            print("   five different values: a seeded estimator draws fresh noise at every call, so the flat saved output has another cause; compare the spread above with the requested precision.")
        else:
            print("   some repeated and some different values:", distinct["seeded, one estimator"], "distinct of 5.")
    except Exception as e:
        print("stage 5 could not be completed:", type(e).__name__, e)

In [ ]:
# 6) the noise experiment of Section 8, with Qiskit's estimator: seeded as in the tutorial, and with a new seed for every evaluation
if HAVE_QISKIT:
    try:
        n_rep, ps = 20, [0.1, 0.01, 0.001]; rows = []; t0 = time.perf_counter()
        ref = table_noise["independent"].set_index("precision p")["mean error (mHa)"]
        for mode in ("seeded estimator, as in the tutorial", "new seed for every evaluation"):
            for p in ps:
                errs = []
                for rep in range(n_rep):
                    if mode.startswith("seeded"):
                        est_p = StatevectorEstimator(default_precision=p, seed=1000 * rep + 17); f = lambda x, est_p=est_p: energy(x, est_p)
                    else:
                        gen = np.random.default_rng(10_000 + rep); f = lambda x, gen=gen, p=p: energy(x, StatevectorEstimator(default_precision=p, seed=int(gen.integers(2 ** 31))))
                    r = minimize(f, [0.0], method="COBYLA", options=dict(maxiter=60, rhobeg=0.3)); errs.append(energy(r.x) - ev2[0])
                errs = np.array(errs); rows.append((mode, p, 1e3 * errs.mean(), 100 * np.mean(errs < CHEMICAL_ACCURACY), float(ref.loc[p])))
        tab = pd.DataFrame(rows, columns=["estimator", "precision p", "mean error (mHa)", "within chem. accuracy (%)", "Part I, independent noise (mHa)"])
        print(tab.round(3).to_string(index=False)); print(f"\ndone in {time.perf_counter() - t0:.0f} s ({n_rep} repeats per row; Part I used 200)")
        fresh = tab[tab.estimator.str.startswith("new seed")].set_index("precision p")["mean error (mHa)"]
        check("with a new seed for every evaluation the error grows with the noise (> 5 mHa at p = 0.1, < 1.5 mHa at p = 0.001), as in Part I", fresh.loc[0.1] > 5 and fresh.loc[0.001] < 1.5)
    except Exception as e:
        print("stage 6 could not be completed:", type(e).__name__, e)

In [ ]:
if HAVE_QISKIT:
    ok = [lab for lab, passed in checks if passed]; bad = [lab for lab, passed in checks if not passed]
    print(f"Part II: {len(ok)} of {len(checks)} checks passed")
    for lab in bad: print("  not passed:", lab)

### What to look for
* **Stages 1 to 4 are agreements.** PySCF's integrals, the 36-term fermionic operator, the 15- and 5-term qubit operators, the UCCSD energy and the one-parameter circuit equal the numbers that Part I derived from Gaussian integrals upward. A failed check points at one of the two implementations; the printed values say which.
* **Stage 5 is the diagnostic.** For the *seeded, one estimator* row, identical values mean that the noise is a constant offset, and Part I's "same draw each call" model describes the tutorial's run; five different values mean that the saved output has another cause. The *new seed for every call* row must always give five different values with a spread of about the requested precision.
* **Stage 6 repeats the noise experiment with Qiskit's estimator.** With a new seed at every evaluation the error grows with $p$ and reaches chemical accuracy only near a milli-Hartree, as in the post; the seeded rows show what the tutorial's code does. To get the behaviour its caption describes, create the estimator without a seed, or with a new seed at every evaluation.